# E8-v2 — Colab CPU validation and execution

This notebook runs the **pretrained FastText Common Crawl `cc.en.300.bin`** retrieval variant. It does not train or fine-tune FastText. The original E8 experiment is not modified.

**Important:** the full binary is multi-GB. The notebook checks free disk and RAM before download. It streams the official gzip artifact to temporary runtime storage, loads it once, and does not place it in Git or Google Drive. The model remains in memory during retrieval.

ToolRet and MCPToolBench++ are evaluated as separate tracks. Primary metrics remain Hit@1, Hit@5, MRR@10 and Macro Hit@1.

Before running: set Colab runtime to CPU (GPU is not used), and make the two dataset roots available in Google Drive.

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, platform, time, gc
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU cores:', os.cpu_count())
print('Disk free GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))
try:
    import psutil
    vm=psutil.virtual_memory()
    print('RAM total GiB:', round(vm.total/1024**3,2), '| available GiB:', round(vm.available/1024**3,2))
except ImportError:
    print('psutil not installed yet')


## 1. Mount Drive and configure dataset paths

Expected ToolRet processed adapter layout:
`manifest.json`, `discovery/tools.json`, `discovery/tasks.json`, `evaluator/task_references.json`.

Expected MCPToolBench++ raw layout: `SOURCES.txt` and `mcptoolbenchpp/*.json`.

Do not put evaluator qrels into discovery directories. The adapter rejects evaluator-only fields in discovery data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THESE TWO PATHS to the existing dataset locations in your Drive.
TOOLRET_DIR = Path('/content/drive/MyDrive/taas-data/toolret/processed')
MCP_RAW_DIR = Path('/content/drive/MyDrive/taas-data/raw')

# Results persist in Drive; model binary does not.
RESULTS_DIR = Path('/content/drive/MyDrive/taas-results/E8-CPU-StaticEmbedding-v2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_toolret = [TOOLRET_DIR/'manifest.json', TOOLRET_DIR/'discovery'/'tools.json', TOOLRET_DIR/'discovery'/'tasks.json', TOOLRET_DIR/'evaluator'/'task_references.json']
required_mcp = [MCP_RAW_DIR/'SOURCES.txt', MCP_RAW_DIR/'mcptoolbenchpp']
missing = [str(p) for p in required_toolret + required_mcp if not p.exists()]
if missing:
    raise FileNotFoundError('Dataset paths are not ready. Fix TOOLRET_DIR / MCP_RAW_DIR:\n'+'\n'.join(missing))
print('Dataset roots validated.')


## 2. Clone the exact PR branch and install CPU dependencies

The notebook checks out the E8-v2 PR branch, not the default branch. No benchmark dataset or model artifact is committed.

In [ ]:
REPO_URL = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'feature/e8-cpu-fasttext-v2'
REPO_DIR = Path('/content/taas-provenance-discovery')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO_DIR)],check=True)
%cd /content/taas-provenance-discovery
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','pytest','fasttext-wheel==0.9.2'],check=True)
print('Checked out:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


## 3. Run unit tests and syntax/config checks

These checks validate the implementation mechanics only; they do not establish retrieval quality or prove that the real pretrained model loads successfully.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_e8_cpu_static_embedding.py'],check=True)
subprocess.run([sys.executable,'-m','py_compile','experiments/e8_cpu_static_embedding.py'],check=True)
subprocess.run([sys.executable,'-m','json.tool','docs/experiment_design/E8_CPU_STATIC_EMBEDDING_v2.json'],stdout=subprocess.DEVNULL,check=True)
print('Unit tests, Python compilation and JSON validation passed.')


## 4. Resource gate and load pretrained model (inference only)

The official Common Crawl model is large. The 8 GiB free-disk threshold is a safety gate, not a guarantee that RAM is sufficient. If available RAM is below 10 GiB, stop and select a larger Colab runtime rather than risking an OOM. Do not switch to a different model silently.

In [ ]:
import psutil
free_gib = shutil.disk_usage('/content').free / 1024**3
available_ram_gib = psutil.virtual_memory().available / 1024**3
print(f'Free disk: {free_gib:.2f} GiB | Available RAM: {available_ram_gib:.2f} GiB')
if free_gib < 8.0:
    raise RuntimeError('STOP: less than 8 GiB free disk for streamed model download.')
if available_ram_gib < 10.0:
    raise RuntimeError('STOP: less than 10 GiB available RAM. Choose a higher-RAM CPU runtime.')

from experiments.e8_cpu_static_embedding import prepare_pretrained_model
load_started = time.perf_counter()
fasttext_model, model_sha256, model_metadata = prepare_pretrained_model(
    model_path=None, cache_dir=None, min_free_gb=8.0
)
load_seconds = time.perf_counter() - load_started
print(json.dumps({'model':model_metadata,'sha256':model_sha256,'load_seconds':load_seconds,
                   'available_ram_after_load_gib':round(psutil.virtual_memory().available/1024**3,2)},indent=2))
assert fasttext_model.get_dimension() == 300
probe = fasttext_model.get_word_vector('unseen_mcp_tool_identifier_987654')
assert len(probe) == 300 and all(map(lambda x: float(x)==float(x), probe))
print('300d model and subword/OOV vector probe passed.')


## 5. Load both benchmark tracks and inspect denominators

The evaluator labels are consumed only by the metric evaluator after rankings have been generated. The IDF model is fitted only over candidate tool documents, independently per track.

In [ ]:
from experiments.e8_retrieval_campaign import load_toolret_track, load_mcptoolbenchpp_track
toolret = load_toolret_track(TOOLRET_DIR)
mcpt = load_mcptoolbenchpp_track(MCP_RAW_DIR)
for t in (toolret, mcpt):
    print(json.dumps({'track':t.dataset_id,'tasks':len(t.tasks),'candidate_tools':len(t.tools),
                      'qrel_tasks':sum(bool(t.qrels.get(x.task_id)) for x in t.tasks),
                      'exclusions':t.exclusions},indent=2))


## 6. Run 100-query smoke tests independently

Run this gate first. It uses the complete candidate tool universe but only the first 100 discovery queries per track. It writes smoke-test artifacts to Drive, separate from the full-run directory.

In [ ]:
from dataclasses import replace
from experiments.e8_cpu_static_embedding import run_track
SMOKE_DIR = RESULTS_DIR/'smoke_100'
SMOKE_DIR.mkdir(parents=True,exist_ok=True)
def first_n_tasks(track, n=100):
    tasks=track.tasks[:n]
    ids={t.task_id for t in tasks}
    return replace(track, tasks=tasks,
                   qrels={k:v for k,v in track.qrels.items() if k in ids},
                   cluster_by_task={k:v for k,v in track.cluster_by_task.items() if k in ids},
                   secondary_qrels={k:v for k,v in track.secondary_qrels.items() if k in ids})

smoke_summaries=[]
for track in (toolret, mcpt):
    smoke=first_n_tasks(track,100)
    out=SMOKE_DIR/track.dataset_id
    result=run_track(smoke,fasttext_model,out,model_sha256,{'top_k':10,'checkpoint_tasks':16})
    summary={'track':track.dataset_id,'smoke_tasks':len(smoke.tasks),'candidate_tools':len(smoke.tools),
             'primary_metrics':result['metrics_primary_qrels']}
    smoke_summaries.append(summary)
    print(json.dumps(summary,indent=2))
(SMOKE_DIR/'SMOKE_SUMMARY.json').write_text(json.dumps(smoke_summaries,indent=2)+'\n')


## 7. Full benchmark — run only after reviewing smoke results

Set `RUN_FULL = True` only after checking the smoke outputs, memory headroom, dataset manifests and expected runtime. This runs each dataset track separately and saves rankings/checkpoints/metrics to Drive.

In [ ]:
RUN_FULL = False  # Explicit opt-in after smoke-test review
if RUN_FULL:
    FULL_DIR = RESULTS_DIR/'full'
    FULL_DIR.mkdir(parents=True,exist_ok=True)
    full_summaries=[]
    for track in (toolret, mcpt):
        out=FULL_DIR/track.dataset_id
        result=run_track(track,fasttext_model,out,model_sha256,{'top_k':10,'checkpoint_tasks':16})
        summary={'track':track.dataset_id,'tasks':len(track.tasks),'candidate_tools':len(track.tools),
                 'primary_metrics':result['metrics_primary_qrels'],
                 'secondary_sensitivity_metrics':result['metrics_secondary_sensitivity_qrels'],
                 'timing_seconds':result['timing_seconds']}
        full_summaries.append(summary)
        print(json.dumps(summary,indent=2))
    (FULL_DIR/'FULL_SUMMARY.json').write_text(json.dumps(full_summaries,indent=2)+'\n')
else:
    print('Full run is disabled. Set RUN_FULL=True only after smoke validation.')


## 8. Export a compact validation record

This record includes the exact Git commit, model hash, runtime telemetry, smoke results and full-run summary if executed. It does not include the model binary.

In [ ]:
validation_record={
 'experiment':'E8-CPU-StaticEmbedding-v2','training_performed':False,
 'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),
 'branch':BRANCH,'model_sha256':model_sha256,'model_metadata':model_metadata,
 'model_load_seconds':load_seconds,'runtime':{'python':sys.version,'platform':platform.platform(),
             'cpu_count':os.cpu_count(),'ram_available_gib_after_load':round(psutil.virtual_memory().available/1024**3,2)},
 'smoke_summary':smoke_summaries,'full_run_enabled':RUN_FULL}
(RESULTS_DIR/'COLAB_VALIDATION_RECORD.json').write_text(json.dumps(validation_record,indent=2)+'\n')
print('Validation record:', RESULTS_DIR/'COLAB_VALIDATION_RECORD.json')
print(json.dumps(validation_record,indent=2))


## Scientific and operational cautions

- The smoke test is a pipeline check, not a statistically powered comparison.
- Do not pool ToolRet and MCPToolBench++ metrics.
- Do not tune IDF, preprocessing, or thresholds on evaluator qrels.
- The model hash, dataset manifests, query/tool counts and exclusions must accompany any reported result.
- The notebook does not merge the PR or alter the original E8 run.

Official model documentation: https://fasttext.cc/docs/en/crawl-vectors
